# Phase 4a - WiMANS + CSI-Bench count features for the final ESP32 model (Kaggle)

Turns the two Kaggle-only datasets into the **same 10 window cues** the ESP32 counter uses
(`esp32_counter.py`), one row per clip x link, so the laptop notebook `phase4_final_local.ipynb`
can pool them with mat80 and WiCount.

| Dataset | Used | Clip | Links | People |
|---|---|---|---|---|
| WiMANS (Intel 5300) | 5 GHz only (2.4 GHz is white noise) | 3 s, 30 subcarriers -> 52 tones | 9 (3 tx x 3 rx) | 0-5, 3 rooms |
| CSI-Bench Localization | 168-subcarrier recordings | 5 s, 3 x 56 subcarriers -> 52 tones | 3 devices | 1-2, several rooms |

The clips are too short for the device's 30 s causal baseline, so each tone is divided by its mean over the
clip. The laptop notebook measures how much that changes the cues and the count on mat80 / WiCount.

**Kaggle settings:** inputs `shuokanghuang/wimans` and `guozhenjennzhu/csi-bench`, CPU is enough.
Output: `/kaggle/working/wimans_clip_features.parquet` and `csibench_clip_features.parquet` (a few MB).
Download them to `D:/wifi/Kaggle results/phase4_features/`.

In [ ]:
# 1. Setup and paths
import copy
import glob
import json
import os
import time
from multiprocessing import get_context

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from sklearn.metrics import f1_score

# Locate dataset folders under /kaggle/input (works for any dataset slug / nesting)
SKIP_DIRS = {"amp", "mat", "video"}            # huge leaf folders: never walk into them


def find_dirs(test, root="/kaggle/input", max_depth=6):
    """Return folders for which test(folder, file_names) is True (shallow walk)."""
    hits = []
    base = root.rstrip("/").count("/")
    for folder, subdirs, files in os.walk(root):
        if test(folder, files):
            hits.append(folder)
        if folder.count("/") - base >= max_depth:
            subdirs[:] = []
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS and not s.startswith("sub_")]
    return sorted(hits)


print("attached datasets:", sorted(os.listdir("/kaggle/input")))

roots = find_dirs(lambda folder, files: "annotation.csv" in files)
assert roots, "annotation.csv not found - is the WiMANS dataset attached?"
WIMANS_ROOT = roots[0]
ANNOT = os.path.join(WIMANS_ROOT, "annotation.csv")
AMP_DIR = os.path.join(WIMANS_ROOT, "wifi_csi", "amp")
if not os.path.isdir(AMP_DIR):
    hits = glob.glob(os.path.join(WIMANS_ROOT, "**", "amp"), recursive=True)
    AMP_DIR = hits[0]

OUT_ROOT = "/kaggle/working/shared_format"
SHARED = os.path.join(OUT_ROOT, "wimans")
RESULTS = "/kaggle/working/results/wimans"
os.makedirs(SHARED, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

# ---- shared-format settings (identical to the local-dataset notebooks) ----
FS = 50              # common sample rate (Hz)
MAX_GAP_BINS = 3     # gaps up to 60 ms are interpolated
MAX_MISSING = 0.20   # samples/windows with more missing bins are dropped
N_TONES = 52
WIN_S = 3.0          # one WiMANS sample = one window
HOP_S = 3.0
WIMANS_RATE = 1000   # nominal packets/s (amp files carry no timestamps)
BANDS_USED = ["5"]   # 2.4 GHz is white noise in WiMANS (see the Phase 0 signal probe)
N_BINS = int(WIN_S * FS)
N_WORKERS = os.cpu_count()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
print("WiMANS root:", WIMANS_ROOT)
print("amp dir    :", AMP_DIR, "  files:", len(os.listdir(AMP_DIR)))
print("device     :", DEVICE, "  cpus:", N_WORKERS)

In [ ]:
# 2. Phase 0 - labels
ann = pd.read_csv(ANNOT, dtype=str)
print(ann.shape)
print(list(ann.columns))

ann["count"] = ann["number_of_users"].astype(int)
ann["session"] = ann["label"].str.split("_").str[1]          # act_<session>_<sample>
ann["has_file"] = [os.path.exists(os.path.join(AMP_DIR, l + ".npy")) for l in ann["label"]]
print("annotation rows with an amp file:", ann["has_file"].sum(), "/", len(ann))

display(pd.crosstab([ann["environment"], ann["wifi_band"]], ann["count"], margins=True))

# how many sessions per environment/band, and are counts mixed inside a session?
per_session = ann.groupby("session").agg(env=("environment", "first"), band=("wifi_band", "first"),
                                         n=("label", "size"), counts=("count", "nunique"),
                                         envs=("environment", "nunique"), bands=("wifi_band", "nunique"))
print("sessions:", len(per_session), "| sessions spanning >1 env:", (per_session["envs"] > 1).sum(),
      "| >1 band:", (per_session["bands"] > 1).sum(), "| with >1 count:", (per_session["counts"] > 1).sum())
display(per_session.groupby(["env", "band"]).agg(sessions=("n", "size"), samples=("n", "sum")))

act_cols = [c for c in ann.columns if c.endswith("_activity")]
print("activities:", sorted(set(ann[act_cols].stack().dropna()) - {"nan"}))

In [ ]:
# 3. Shared-format helpers: sub-bands, 50 Hz grid, normalisation, count cues

# ---------------------------------------------------------------
# 20 MHz sub-bands of an 80 MHz channel (256-point FFT, natural order)
# ---------------------------------------------------------------
# Sub-band s covers tone indices -128+64s ... -65+64s. We keep 52 tones,
# skipping 6 at each edge. This automatically avoids DC (bins 0, 1, 255)
# and the guard band (bins 123-133) that carry no channel data.
def subband_bins(s):
    start = -128 + 64 * s
    tones = np.arange(start + 6, start + 6 + N_TONES)
    return np.mod(tones, 256)


SUBBANDS = [subband_bins(s) for s in range(4)]


# ---------------------------------------------------------------
# Normalisation
# ---------------------------------------------------------------
def gain_normalise(A):
    """Per packet: divide by the median over tones (removes AGC jumps)."""
    return A / (np.median(A, axis=1, keepdims=True) + 1e-6)


def recording_normalise(X):
    """Per recording + receiver: each tone relative to its mean -> fluctuation around 0."""
    mu = np.nanmean(X, axis=0, keepdims=True)
    dead = mu < 1e-3 * np.nanmedian(mu)                    # null tones, if any
    Y = X / (mu + 1e-6) - 1.0
    Y[:, dead[0]] = 0.0
    return Y


# ---------------------------------------------------------------
# Resampling onto the 50 Hz grid
# ---------------------------------------------------------------
def resample_to_grid(t, A):
    """t: packet times (s, sorted), A: (packets, tones).
    Returns X (bins, tones) with NaN rows for empty bins, and the valid mask."""
    idx = np.floor((t - t[0]) * FS).astype(np.int64)
    n_bins = int(idx[-1]) + 1

    uniq, first = np.unique(idx, return_index=True)        # idx is sorted
    sums = np.add.reduceat(A.astype(np.float64), first, axis=0)
    counts = np.diff(np.append(first, len(idx)))

    X = np.full((n_bins, A.shape[1]), np.nan, dtype=np.float32)
    X[uniq] = sums / counts[:, None]
    valid = np.zeros(n_bins, dtype=bool)
    valid[uniq] = True
    return X, valid


def fill_short_gaps(X, valid):
    """Linear interpolation over gaps of at most MAX_GAP_BINS; longer gaps stay NaN."""
    X, valid = X.copy(), valid.copy()
    missing = np.flatnonzero(~valid)
    if len(missing) == 0:
        return X, valid

    runs = np.split(missing, np.flatnonzero(np.diff(missing) > 1) + 1)
    for run in runs:
        left, right = run[0] - 1, run[-1] + 1
        if len(run) <= MAX_GAP_BINS and left >= 0 and right < len(X):
            w = ((run - left) / (right - left))[:, None]
            X[run] = X[left] * (1 - w) + X[right] * w
            valid[run] = True
    return X, valid


def to_shared_series(t, A):
    """Raw amplitude of ONE view -> normalised 50 Hz series (bins, 52) + valid mask."""
    A = gain_normalise(A.astype(np.float32))
    X, valid = resample_to_grid(t, A)
    X, valid = fill_short_gaps(X, valid)
    return recording_normalise(X), valid


# ---------------------------------------------------------------
# Count cues per window
# ---------------------------------------------------------------
FEATURE_NAMES = [
    "cv_median",       # median over tones of the fluctuation std
    "cv_p90",          # 90th percentile of the same
    "cv_p10",          # 10th percentile of the same
    "diff_energy",     # mean |x(t) - x(t-1)|  (motion energy)
    "pc1_frac",        # share of variance in the first principal component
    "n_pc_90",         # components needed for 90 % of the variance
    "tone_corr",       # mean correlation between tones
    "low_freq_frac",   # temporal power below 2 Hz (slow body motion)
    "high_freq_frac",  # temporal power above 5 Hz
    "mean_std",        # std of the tone-averaged series
]


def window_features(W):
    """W: (bins, 52) normalised window. Missing bins count as 'no change' (0)."""
    W = np.nan_to_num(W.astype(np.float32))
    sd = W.std(axis=0)
    diff_energy = np.abs(np.diff(W, axis=0)).mean()

    Xc = W - W.mean(axis=0)
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    n90 = np.searchsorted(np.cumsum(frac), 0.90) + 1

    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(Xc.T)
    tone_corr = np.nanmean(C[np.triu_indices_from(C, k=1)])

    x = Xc.mean(axis=1)
    P = np.abs(np.fft.rfft(x)) ** 2
    f = np.fft.rfftfreq(len(x), 1.0 / FS)
    p_total = P[f > 0].sum() + 1e-12
    low = P[(f > 0) & (f < 2.0)].sum() / p_total
    high = P[f > 5.0].sum() / p_total

    return np.array([np.median(sd), np.percentile(sd, 90), np.percentile(sd, 10),
                     diff_energy, frac[0], n90, tone_corr, low, high, x.std()],
                    dtype=np.float32)


# ---------------------------------------------------------------
# Windowing + saving one recording
# ---------------------------------------------------------------
def save_recording(rec):
    """rec: dict with series (views, bins, 52), valid (bins,), view_names, metadata."""
    S, valid = rec["series"], rec["valid"]
    win, hop = int(WIN_S * FS), int(HOP_S * FS)

    starts = [a for a in range(0, S.shape[1] - win + 1, hop)
              if valid[a:a + win].mean() >= 1 - MAX_MISSING]

    F = np.zeros((S.shape[0], len(starts), len(FEATURE_NAMES)), dtype=np.float32)
    for v in range(S.shape[0]):
        for j, a in enumerate(starts):
            F[v, j] = window_features(S[v, a:a + win])

    meta = {k: rec[k] for k in ["dataset", "recording", "group", "setup", "receiver", "count"]}
    meta["seconds"] = round(S.shape[1] / FS, 1)
    meta["missing_frac"] = round(float(1 - valid.mean()), 4)
    meta["n_windows"] = len(starts)

    out_dir = os.path.join(OUT_ROOT, rec["dataset"])
    os.makedirs(out_dir, exist_ok=True)
    np.savez_compressed(os.path.join(out_dir, rec["recording"] + ".npz"),
                        series=S.astype(np.float16), valid=valid,
                        starts=np.array(starts, dtype=np.int32), feats=F,
                        view_names=np.array(rec["view_names"]), meta=json.dumps(meta))
    return meta


def already_done(dataset, recording):
    return os.path.exists(os.path.join(OUT_ROOT, dataset, recording + ".npz"))


print("sub-band bins:", [(int(b[0]), int(b[-1])) for b in SUBBANDS])


# Map any number of subcarriers onto the 52-tone grid (linear interpolation over frequency)
def tone_interp_matrix(n_src, n_dst=N_TONES):
    src = np.linspace(0.0, 1.0, n_src)
    dst = np.linspace(0.0, 1.0, n_dst)
    W = np.zeros((n_dst, n_src), dtype=np.float32)
    for i, x in enumerate(dst):
        j = min(np.searchsorted(src, x, side="right") - 1, n_src - 2)
        w = (x - src[j]) / (src[j + 1] - src[j])
        W[i, j] = 1.0 - w
        W[i, j + 1] = w
    return W

In [ ]:
# 4. Phase 1 - convert every 5 GHz sample to the shared format (9 views x 150 bins x 52 tones)
W30 = tone_interp_matrix(30)
labels = ann.loc[ann["has_file"] & ann["wifi_band"].isin(BANDS_USED), "label"].tolist()
print("samples to convert:", len(labels), "bands:", BANDS_USED)


def convert_sample(label):
    A = np.load(os.path.join(AMP_DIR, label + ".npy")).astype(np.float32)   # (T, 3, 3, 30)
    t = np.arange(A.shape[0]) / WIMANS_RATE
    series = np.full((9, N_BINS, N_TONES), np.nan, dtype=np.float32)
    valid = np.zeros(N_BINS, dtype=bool)
    feats = np.zeros((9, len(FEATURE_NAMES)), dtype=np.float32)

    for v in range(9):
        a, b = divmod(v, 3)
        X, ok = to_shared_series(t, A[:, a, b, :] @ W30.T)
        n = min(len(X), N_BINS)
        series[v, :n] = X[:n]
        valid[:n] = ok[:n]
        feats[v] = window_features(series[v])
    return label, series.astype(np.float16), valid, feats


N = len(labels)
SERIES = np.zeros((N, 9, N_BINS, N_TONES), dtype=np.float16)       # ~0.8 GB for 5 GHz
VALID = np.zeros((N, N_BINS), dtype=bool)
FEATS = np.zeros((N, 9, len(FEATURE_NAMES)), dtype=np.float32)
pos = {l: i for i, l in enumerate(labels)}

t0 = time.time()
with get_context("fork").Pool(N_WORKERS) as pool:
    for k, (label, s, ok, f) in enumerate(pool.imap_unordered(convert_sample, labels, chunksize=16)):
        i = pos[label]
        SERIES[i], VALID[i], FEATS[i] = s, ok, f
        if k % 1000 == 0:
            print(f"{k:6d} / {N}   [{time.time() - t0:.0f} s]", flush=True)

meta = ann.set_index("label").loc[labels].reset_index()
meta["valid_frac"] = VALID.mean(1)
np.save(os.path.join(SHARED, "series.npy"), SERIES)
np.save(os.path.join(SHARED, "feats.npy"), FEATS)
meta.to_parquet(os.path.join(SHARED, "meta.parquet"), index=False)
with open(os.path.join(SHARED, "format.json"), "w") as f:
    json.dump({"fs_hz": FS, "win_s": WIN_S, "n_tones": N_TONES, "views": "tx*3+rx (9 links)",
               "tone_mapping": "30 Intel-5300 subcarriers linearly interpolated to 52",
               "feature_names": FEATURE_NAMES}, f, indent=1)
print(f"done in {time.time() - t0:.0f} s; samples with >= 80 % valid bins:",
      int((meta["valid_frac"] >= 1 - MAX_MISSING).sum()), "/", N)

In [ ]:
# 6. WiMANS -> one feature row per (clip, link); session + clip number kept for the empty-room calibration
import shutil

EXPORT = "/kaggle/working"
LINK_NAMES_W = [f"tx{a}_rx{b}" for a in range(3) for b in range(3)]

keep = meta["valid_frac"].values >= 1 - MAX_MISSING
idx = np.flatnonzero(keep)
n_links = len(LINK_NAMES_W)

wim = pd.DataFrame({"clip": np.repeat(idx, n_links), "view": np.tile(LINK_NAMES_W, len(idx))})
wim["dataset"] = "wimans"
wim["recording"] = "wimans_" + meta["label"].values[wim["clip"]]
wim["setup"] = meta["environment"].values[wim["clip"]]
wim["session"] = meta["session"].values[wim["clip"]]
wim["sample_no"] = pd.to_numeric(meta["label"].str.split("_").str[2], errors="coerce").values[wim["clip"]]
wim["count"] = meta["count"].values[wim["clip"]].astype(int)
wim["window_s"] = WIN_S
wim[FEATURE_NAMES] = FEATS[idx].reshape(-1, len(FEATURE_NAMES))
wim = wim.drop(columns="clip")

wim.to_parquet(os.path.join(EXPORT, "wimans_clip_features.parquet"), index=False)
print(f"WiMANS: {wim['recording'].nunique()} clips, {len(wim)} rows")
display(pd.crosstab(wim.drop_duplicates("recording")["setup"], wim.drop_duplicates("recording")["count"],
                    margins=True))

# ---- free memory + disk before CSI-Bench (the raw series are not needed any more) ----
del SERIES, VALID
shutil.rmtree("/kaggle/working/shared_format", ignore_errors=True)
shutil.rmtree("/kaggle/working/results", ignore_errors=True)

In [ ]:
# 1. Setup and task discovery
import copy
import glob
import json
import os
import re
import time
from multiprocessing import get_context

import h5py
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from sklearn.metrics import f1_score

# Locate dataset folders under /kaggle/input (works for any dataset slug / nesting)
SKIP_DIRS = {"amp", "mat", "video"}            # huge leaf folders: never walk into them


def find_dirs(test, root="/kaggle/input", max_depth=6):
    """Return folders for which test(folder, file_names) is True (shallow walk)."""
    hits = []
    base = root.rstrip("/").count("/")
    for folder, subdirs, files in os.walk(root):
        if test(folder, files):
            hits.append(folder)
        if folder.count("/") - base >= max_depth:
            subdirs[:] = []
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS and not s.startswith("sub_")]
    return sorted(hits)


print("attached datasets:", sorted(os.listdir("/kaggle/input")))

TASK_DIRS = find_dirs(lambda folder, files: os.path.exists(
    os.path.join(folder, "metadata", "sample_metadata.csv")), max_depth=7)
assert TASK_DIRS, "no <task>/metadata/sample_metadata.csv found - is CSI-Bench attached?"
for d in TASK_DIRS:
    print(d)

OUT_ROOT = "/kaggle/working/shared_format"
AUDIT = "/kaggle/working/csibench_audit"
SHARED = os.path.join(OUT_ROOT, "csibench_esp32")
SHARED_LOC = os.path.join(OUT_ROOT, "csibench_loc")
RESULTS = "/kaggle/working/results/csibench"
for folder in [AUDIT, SHARED, SHARED_LOC, RESULTS]:
    os.makedirs(folder, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
print("device:", DEVICE)

# ---- shared-format settings (identical to the local-dataset notebooks) ----
FS = 50              # common sample rate (Hz)
MAX_GAP_BINS = 3     # gaps up to 60 ms are interpolated
MAX_MISSING = 0.20   # samples/windows with more missing bins are dropped
N_TONES = 52
WIN_S = 5.0                   # CSI-Bench samples are 5 s (10 s for breathing)
HOP_S = 5.0
DEVICE_PATTERN = "esp32"      # case-insensitive match on the device name
ASSUMED_RATE_HZ = 100         # checked against the sample lengths in cell 3
N_WORKERS = os.cpu_count()

PATH_FIELDS = re.compile(r"user_(?P<user>[^/]+)/act_(?P<act>[^/]+)/env_(?P<env>[^/]+)/"
                         r"device_(?P<device>[^/]+)/session_(?P<session>[^/]+?)__freq(?P<freq>[^/]+?)\.h5$")


def read_amps(f):
    """CSI-Bench stores CSI_amps as (subcarriers, time, 1). Return (time, subcarriers)."""
    key = "CSI_amps" if "CSI_amps" in f else sorted(f.keys())[0]
    a = np.asarray(f[key], dtype=np.float32)
    if a.ndim == 3:
        a = a[:, :, 0]
    return a.T


def from_sub_folder(p):
    """Cut a metadata file_path down to the part starting at its first 'sub_*' folder."""
    parts = p.replace("\\", "/").split("/")
    for i, part in enumerate(parts):
        if part.startswith("sub_"):
            return "/".join(parts[i:])
    return p.lstrip("./")


def resolve_paths(task_dir, rel_paths, n_test=20):
    """Find which folder (the task folder or one of its parents) the file paths are relative to."""
    tail = [from_sub_folder(p) for p in rel_paths]
    base = task_dir
    while base.count("/") > 2:
        hits = sum(os.path.exists(os.path.join(base, t)) for t in tail[:n_test])
        if hits >= max(1, min(n_test, len(tail)) // 2):
            return [os.path.join(base, t) for t in tail], base
        base = os.path.dirname(base)
    print("  WARNING: could not resolve file paths for", task_dir)
    return [os.path.normpath(os.path.join(task_dir, p)) for p in rel_paths], None

In [ ]:
# 2. Phase 0 - metadata, labels, devices and splits per task
def task_name(d):
    parts = d.rstrip("/").split("/")
    return parts[-1] if parts[-2] != "Multitask" else "Multitask/" + parts[-1]


all_meta, audit = [], {}
for d in TASK_DIRS:
    name = task_name(d)
    md = pd.read_csv(os.path.join(d, "metadata", "sample_metadata.csv"))
    print("=" * 100)
    print(name, md.shape)
    print("columns:", list(md.columns))
    display(md.head(3))

    # fields encoded in the file path (user / activity / environment / device / session / freq)
    fp = md["file_path"].astype(str) if "file_path" in md.columns else pd.Series([""] * len(md))
    parsed = fp.str.extract(PATH_FIELDS)
    for c in parsed.columns:
        if c not in md.columns:
            md[c] = parsed[c]
    md["abs_path"], base = resolve_paths(d, fp.tolist())
    print("  files resolved under:", base)
    md["task"] = name

    # Localization labels are zone bitmasks: number of 1s = number of people
    if name == "Localization":
        md["count"] = md["label"].astype(str).str.count("1")
        print("  people count from label bitmask:")
        display(pd.crosstab(md["count"], md.get("is_multi_subject", "?"), margins=True))

    low_card = [c for c in md.columns if 1 < md[c].nunique() <= 40 and c not in ("abs_path", "file_path")]
    for c in low_card:
        print(f"  {c}: {md[c].value_counts().to_dict()}")

    splits = {}
    for sp in sorted(glob.glob(os.path.join(d, "splits", "*.json"))):
        ids = json.load(open(sp))
        splits[os.path.basename(sp)[:-5]] = len(ids)
    print("  splits:", splits)
    mapping = os.path.join(d, "metadata", "label_mapping.json")
    if os.path.exists(mapping):
        print("  label_mapping:", json.load(open(mapping)))

    audit[name] = {"rows": len(md), "columns": list(md.columns), "splits": splits, "files_base": base,
                   "value_counts": {c: md[c].astype(str).value_counts().to_dict() for c in low_card}}
    all_meta.append(md)

meta = pd.concat(all_meta, ignore_index=True)
if "device" not in meta.columns:
    meta["device"] = "?"
meta["is_esp32"] = meta["device"].astype(str).str.contains(DEVICE_PATTERN, case=False, na=False)
print("=" * 100)
display(pd.crosstab(meta["task"], meta["device"].fillna("?"), margins=True))
with open(os.path.join(AUDIT, "metadata_audit.json"), "w") as f:
    json.dump(audit, f, indent=1, default=str)

In [ ]:
# 6. Localization - layout check: are the 168 subcarriers really 3 devices x 56?
N_BINS = int(WIN_S * FS)                              # one CSI-Bench clip = one 5-s window
LOC_DEVICES = 3
LOC_SUB = 168

loc = meta[meta["task"] == "Localization"].drop_duplicates("abs_path").reset_index(drop=True)
loc["n_sub"] = loc["num_sub"].astype(int)
loc["count"] = loc["count"].astype(int)
print("Localization samples per room / subcarrier count / people:")
display(pd.crosstab([loc["environment"], loc["n_sub"]], loc["count"], margins=True))
display(pd.crosstab(loc["device"], loc["n_sub"]))


def mean_profile(paths):
    """Mean |amplitude| per subcarrier over several files."""
    acc = None
    for p in paths:
        with h5py.File(p, "r") as f:
            m = np.abs(read_amps(f)).mean(axis=0)
        acc = m if acc is None else acc + m
    return acc / len(paths)


profiles = {n: mean_profile(g["abs_path"].sample(min(60, len(g)), random_state=SEED).tolist())
            for n, g in loc.groupby("n_sub")}

fig, axes = plt.subplots(len(profiles), 1, figsize=(14, 3 * len(profiles)))
for ax, (n_sub, prof) in zip(np.atleast_1d(axes), profiles.items()):
    ax.plot(prof)
    for k in range(1, LOC_DEVICES):
        ax.axvline(k * n_sub / LOC_DEVICES, color="r", ls="--")
    ax.set_title(f"mean amplitude per subcarrier, {n_sub} subcarriers (red = assumed device borders)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "loc_subcarrier_profile.png"), dpi=110)
plt.show()

# if the 3 x 56 layout is right, each device block has its own null tones at the same places
blocks = profiles[LOC_SUB].reshape(LOC_DEVICES, -1)
print("correlation between the 3 device blocks' profiles:")
print(np.round(np.corrcoef(blocks), 3))

In [ ]:
# 7. Localization - 168-subcarrier samples -> 3 links x 250 bins x 52 tones + count cues
SUB_PER_DEV = LOC_SUB // LOC_DEVICES
loc_use = loc[loc["n_sub"] == LOC_SUB].reset_index(drop=True)
prof = blocks.mean(axis=0)
KEEP_LOC = prof > 0.02 * np.median(prof)              # drop null tones inside a device block
W_LOC = tone_interp_matrix(int(KEEP_LOC.sum()))
print(len(loc_use), "samples;", int(KEEP_LOC.sum()), "of", SUB_PER_DEV, "tones kept per device")

# Extra cues for "how many separate movers" (all cheap enough for an ESP32)
EXTRA_NAMES = [
    "pc2_frac",          # share of variance in the 2nd principal component
    "pc3_frac",          # ... and in the 3rd (several movers spread variance over components)
    "eig_entropy",       # entropy of the PCA spectrum = effective number of independent motions
    "doppler_centroid",  # mean temporal frequency of the motion (Hz)
    "doppler_spread",    # spread of the temporal frequencies (Hz)
    "psd_entropy",       # flatness of the temporal spectrum (0..1)
    "activity_cv",       # burstiness of motion over 0.2-s segments (one mover: bursty, many: steady)
    "tone_energy_cv",    # how unevenly motion is spread over the 52 tones
    "half_band_corr",    # correlation of motion in the lower vs upper 26 tones
]
LINK_NAMES = [
    "link_env_corr",     # mean correlation of the 9 links' motion envelopes (needs several receivers)
    "link_energy_cv",    # spread of motion energy over the 9 links
]


def extra_features(W):
    """W: (bins, 52) normalised window -> (extra cues, motion envelope over time)."""
    W = np.nan_to_num(W.astype(np.float32))
    Xc = W - W.mean(axis=0)

    # spread of the motion over principal components
    s = np.linalg.svd(Xc, compute_uv=False) ** 2
    frac = s / (s.sum() + 1e-12)
    eig_entropy = -(frac * np.log(frac + 1e-12)).sum()

    # temporal (Doppler) spectrum summed over tones, DC removed
    P = (np.abs(np.fft.rfft(Xc, axis=0)) ** 2).sum(axis=1)[1:]
    f = np.fft.rfftfreq(len(W), 1.0 / FS)[1:]
    p = P / (P.sum() + 1e-12)
    centroid = (p * f).sum()
    spread = np.sqrt((p * (f - centroid) ** 2).sum())
    psd_entropy = -(p * np.log(p + 1e-12)).sum() / np.log(len(p))

    # motion = absolute change between bins
    d = np.abs(np.diff(W, axis=0))
    env = d.mean(axis=1)
    seg = env[:len(env) // 10 * 10].reshape(-1, 10).mean(axis=1)
    activity_cv = seg.std() / (seg.mean() + 1e-9)
    tone_e = d.mean(axis=0)
    tone_energy_cv = tone_e.std() / (tone_e.mean() + 1e-9)

    half = W.shape[1] // 2
    lo, hi = d[:, :half].mean(axis=1), d[:, half:].mean(axis=1)
    half_band_corr = np.corrcoef(lo, hi)[0, 1] if lo.std() > 0 and hi.std() > 0 else 0.0

    cues = np.array([frac[1], frac[2], eig_entropy, centroid, spread, psd_entropy,
                     activity_cv, tone_energy_cv, half_band_corr], dtype=np.float32)
    return cues, env


def link_cues(envs):
    """Cross-link cues from the motion envelopes of all links of one sample."""
    E = np.array(envs)
    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(E)
    energy = E.mean(axis=1)
    return np.array([np.nanmean(C[np.triu_indices(len(E), k=1)]),
                     energy.std() / (energy.mean() + 1e-9)], dtype=np.float32)


def convert_loc(path):
    try:
        with h5py.File(path, "r") as f:
            a = read_amps(f)                                     # (time, 168)
    except Exception:                                            # unreadable file
        return path, None, None, None, None, None

    t = np.arange(len(a)) / ASSUMED_RATE_HZ
    series = np.full((LOC_DEVICES, N_BINS, N_TONES), np.nan, dtype=np.float32)
    valid = np.zeros(N_BINS, dtype=bool)
    feats = np.zeros((LOC_DEVICES, len(FEATURE_NAMES)), dtype=np.float32)
    extra = np.zeros((LOC_DEVICES, len(EXTRA_NAMES)), dtype=np.float32)
    envs = []
    for v in range(LOC_DEVICES):
        block = a[:, v * SUB_PER_DEV:(v + 1) * SUB_PER_DEV][:, KEEP_LOC] @ W_LOC.T
        X, ok = to_shared_series(t, block)
        n = min(len(X), N_BINS)
        series[v, :n] = X[:n]
        valid[:n] = ok[:n]
        feats[v] = window_features(series[v])
        extra[v], e = extra_features(series[v])
        envs.append(e)
    return path, series.astype(np.float16), valid, feats, extra, link_cues(envs)


NL = len(loc_use)
LOC_SERIES = np.zeros((NL, LOC_DEVICES, N_BINS, N_TONES), dtype=np.float16)
LOC_VALID = np.zeros((NL, N_BINS), dtype=bool)
LOC_FEATS = np.zeros((NL, LOC_DEVICES, len(FEATURE_NAMES)), dtype=np.float32)
LOC_EXTRA = np.zeros((NL, LOC_DEVICES, len(EXTRA_NAMES)), dtype=np.float32)
LOC_LINK = np.zeros((NL, len(LINK_NAMES)), dtype=np.float32)
LOC_OK = np.zeros(NL, dtype=bool)
pos = {p: i for i, p in enumerate(loc_use["abs_path"])}

t0 = time.time()
with get_context("fork").Pool(N_WORKERS) as pool:
    results_iter = pool.imap_unordered(convert_loc, loc_use["abs_path"].tolist(), chunksize=32)
    for k, (p, s, v, f, e, l) in enumerate(results_iter):
        if s is not None:
            i = pos[p]
            LOC_SERIES[i], LOC_VALID[i], LOC_FEATS[i] = s, v, f
            LOC_EXTRA[i], LOC_LINK[i], LOC_OK[i] = e, l, True
        if k % 2000 == 0:
            print(f"{k:6d} / {NL}   [{time.time() - t0:.0f} s]", flush=True)

loc_use["converted"] = LOC_OK
loc_use["valid_frac"] = LOC_VALID.mean(1)
np.save(os.path.join(SHARED_LOC, "series.npy"), LOC_SERIES)
np.save(os.path.join(SHARED_LOC, "feats.npy"), LOC_FEATS)
np.save(os.path.join(SHARED_LOC, "extra_feats.npy"), LOC_EXTRA)
np.save(os.path.join(SHARED_LOC, "link_feats.npy"), LOC_LINK)
out_meta = loc_use.rename(columns={"abs_path": "source_path"})
obj_cols = out_meta.select_dtypes("object").columns
out_meta[obj_cols] = out_meta[obj_cols].astype(str)                  # mixed types break parquet
out_meta.to_parquet(os.path.join(SHARED_LOC, "meta.parquet"), index=False)
with open(os.path.join(SHARED_LOC, "format.json"), "w") as f:
    json.dump({"fs_hz": FS, "win_s": WIN_S, "n_tones": N_TONES, "assumed_rate_hz": ASSUMED_RATE_HZ,
               "views": "3 device links (168 = 3 x 56 subcarriers)", "tones_kept_per_device": int(KEEP_LOC.sum()),
               "feature_names": FEATURE_NAMES, "extra_names": EXTRA_NAMES, "link_names": LINK_NAMES}, f, indent=1)
print(f"converted {LOC_OK.sum()} / {NL} in {time.time() - t0:.0f} s;",
      "with >= 80 % valid bins:", int((loc_use["valid_frac"] >= 1 - MAX_MISSING).sum()))

In [ ]:
# 9. CSI-Bench Localization -> one feature row per (clip, device link)
keep = LOC_OK & (loc_use["valid_frac"].values >= 1 - MAX_MISSING)
idx = np.flatnonzero(keep)
LINK_NAMES_C = [f"dev{v}" for v in range(LOC_DEVICES)]

cb = pd.DataFrame({"clip": np.repeat(idx, LOC_DEVICES), "view": np.tile(LINK_NAMES_C, len(idx))})
cb["dataset"] = "csibench"
cb["recording"] = "csibench_" + pd.Series(idx).astype(str).str.zfill(6).repeat(LOC_DEVICES).values
cb["setup"] = loc_use["environment"].astype(str).values[cb["clip"]]
session_col = "session" if "session" in loc_use.columns else "abs_path"   # parsed from the file path
cb["session"] = loc_use[session_col].astype(str).values[cb["clip"]]
cb["sample_no"] = cb["clip"]
cb["count"] = loc_use["count"].values[cb["clip"]].astype(int)
cb["window_s"] = WIN_S
cb[FEATURE_NAMES] = LOC_FEATS[idx].reshape(-1, len(FEATURE_NAMES))
cb = cb.drop(columns="clip")

cb.to_parquet(os.path.join(EXPORT, "csibench_clip_features.parquet"), index=False)
print(f"CSI-Bench: {cb['recording'].nunique()} clips, {len(cb)} rows")
display(pd.crosstab(cb.drop_duplicates("recording")["setup"], cb.drop_duplicates("recording")["count"],
                    margins=True))

In [ ]:
# 10. Keep only the two feature tables in the output (small download)
for name in ["shared_format", "csibench_audit", "results"]:
    shutil.rmtree(os.path.join(EXPORT, name), ignore_errors=True)

for name in sorted(os.listdir(EXPORT)):
    path = os.path.join(EXPORT, name)
    if os.path.isfile(path):
        print(f"{name:40s} {os.path.getsize(path) / 1e6:8.2f} MB")